# 1. Introduction

This notebook prepares the demographic information contained in the `patients.csv` file from the processed Parkinson's Disease Smartwatch Dataset (PADS). The dataset includes participant identifiers, diagnostic labels, demographic characteristics, family-history information, and selected clinical variables.

The purpose of this notebook is to standardize categorical values, validate numerical variables, identify possible data-quality issues, and generate a cleaned demographic table for subsequent statistical analysis and machine learning.

## Objectives

This notebook aims to:

- Load the processed `patients.csv` dataset.
- Inspect the structure and completeness of the demographic data.
- Standardize categorical variables.
- Validate age, age at diagnosis, height, and weight.
- Create standardized diagnostic groups.
- Flag duplicate participant identifiers.
- Exclude sensitive free-text clinical comments.
- Export the cleaned demographic dataset and QA summary.


In [1]:
# =============================================================================
# Libraries
# =============================================================================

from pathlib import Path
import pandas as pd

# 2. Loading the Demographic Data

The demographic data are loaded from the interim `patients.csv` file. The relative path below assumes that this notebook is stored in the `notebooks` folder and the source dataset is stored in `data/interim`. Cleaned outputs are saved to `data/processed`.


In [2]:
# Define input and output paths
input_file = Path("../data/interim/patients.csv")
output_folder = Path("../data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

print(f"Input file: {input_file.resolve()}")
print(f"File exists: {input_file.exists()}")

if not input_file.exists():
    raise FileNotFoundError(
        f"Could not find {input_file}. "
        "Please ensure that patients.csv is located in the data/interim directory."
    )

Input file: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\interim\patients.csv
File exists: True


In [3]:
# Load the demographic dataset
df = pd.read_csv(input_file)

print(f"Number of records: {len(df)}")
print(f"Number of columns: {df.shape[1]}")

df.head()

Number of records: 469
Number of columns: 14


,patient_id,study_id,condition,label,disease_comment,age_at_diagnosis,age,height_cm,weight_kg,gender,handedness,appearance_in_kinship,appearance_in_first_grade_kinship,effect_of_alcohol_on_tremor
0,1,PADS,Healthy,0,-,56,56,173,78,male,right,True,True,Unknown
1,2,PADS,Other Movement Disorders,2,Left-Sided resting tremor and hypokinesia with...,69,81,193,104,male,right,False,NaN,No effect
2,3,PADS,Healthy,0,-,45,45,170,78,female,right,False,NaN,Unknown
3,4,PADS,Parkinson's,1,IPS akinetic-rigid type,63,67,161,90,female,right,False,NaN,No effect
4,5,PADS,Parkinson's,1,IPS tremordominant type,65,75,172,86,male,left,False,NaN,Unknown


# 3. Initial Data Inspection

The dataset is reviewed to confirm its dimensions, variable names, data types, and missing values before cleaning.


In [4]:
# Display column names and data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 469 entries, 0 to 468
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   patient_id                         469 non-null    int64 
 1   study_id                           469 non-null    str   
 2   condition                          469 non-null    str   
 3   label                              469 non-null    int64 
 4   disease_comment                    469 non-null    str   
 5   age_at_diagnosis                   469 non-null    int64 
 6   age                                469 non-null    int64 
 7   height_cm                          469 non-null    int64 
 8   weight_kg                          469 non-null    int64 
 9   gender                             469 non-null    str   
 10  handedness                         469 non-null    str   
 11  appearance_in_kinship              469 non-null    bool  
 12  appearance_in_first

In [5]:
# Summarize missing values
missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_values")
)

missing_summary.head(15)

,missing_values
appearance_in_first_grade_kinship,288
patient_id,0
condition,0
label,0
disease_comment,0
study_id,0
age_at_diagnosis,0
age,0
weight_kg,0
height_cm,0


# 4. Standardizing Categorical Variables

Categorical variables are standardized to improve consistency. Gender and handedness values are cleaned by removing extra spaces and applying consistent capitalization. Family-history variables are converted to `Yes`, `No`, or `Unknown`, while missing alcohol-effect values are labelled as `Unknown`.


In [6]:
# Preserve the original condition label for traceability
df["condition_original"] = df["condition"]

# Standardize gender and handedness
df["gender"] = df["gender"].astype("string").str.strip().str.title()
df["handedness"] = df["handedness"].astype("string").str.strip().str.title()

# Standardize Yes/No variables
yes_no_map = {
    True: "Yes",
    False: "No",
    "True": "Yes",
    "False": "No",
    "Yes": "Yes",
    "No": "No",
}

df["family_history_any"] = (
    df["appearance_in_kinship"]
    .map(yes_no_map)
    .fillna("Unknown")
)

df["family_history_first_degree"] = (
    df["appearance_in_first_grade_kinship"]
    .map(yes_no_map)
    .fillna("Unknown")
)

df["alcohol_effect_on_tremor"] = (
    df["effect_of_alcohol_on_tremor"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.title()
)

# 5. Standardizing Diagnostic Groups

The numerical diagnosis labels are mapped to three standardized study groups: Healthy Control, Parkinson's Disease, and Other Movement Disorder.


In [7]:
condition_map = {
    0: "Healthy Control",
    1: "Parkinson's Disease",
    2: "Other Movement Disorder",
}

df["condition_group"] = df["label"].map(condition_map)

df[["label", "condition_group"]].drop_duplicates().sort_values("label")

,label,condition_group
0,0,Healthy Control
3,1,Parkinson's Disease
1,2,Other Movement Disorder


# 6. Validating Numerical Variables

Age, age at diagnosis, height, and weight are converted to numeric values. Values outside predefined plausible ranges are replaced with missing values. Age at diagnosis is also set to missing for healthy controls because it is not applicable.


In [8]:
numeric_columns = [
    "age",
    "age_at_diagnosis",
    "height_cm",
    "weight_kg",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Replace implausible values with missing values
df.loc[~df["age"].between(18, 100), "age"] = pd.NA
df.loc[~df["height_cm"].between(120, 230), "height_cm"] = pd.NA
df.loc[~df["weight_kg"].between(30, 250), "weight_kg"] = pd.NA

# Diagnosis age is not applicable to healthy controls
df.loc[
    df["condition_group"] == "Healthy Control",
    "age_at_diagnosis"
] = pd.NA

# Remove invalid diagnosis ages
df.loc[
    (df["age_at_diagnosis"] <= 0)
    | (df["age_at_diagnosis"] > df["age"]),
    "age_at_diagnosis"
] = pd.NA

# 7. Quality Assurance Checks

Duplicate participant identifiers are flagged for review. Only the variables required for the cleaned demographic table are retained, while free-text clinical comments are excluded to reduce sensitivity and prevent possible target leakage.


In [9]:
# Flag duplicate participant IDs
df["duplicate_patient_id"] = df["patient_id"].duplicated(keep=False)

columns_to_keep = [
    "patient_id",
    "study_id",
    "condition_original",
    "condition_group",
    "label",
    "age",
    "age_at_diagnosis",
    "height_cm",
    "weight_kg",
    "gender",
    "handedness",
    "family_history_any",
    "family_history_first_degree",
    "alcohol_effect_on_tremor",
    "duplicate_patient_id",
]

clean_df = df[columns_to_keep].copy()

clean_df.head()

,patient_id,study_id,condition_original,condition_group,label,age,age_at_diagnosis,height_cm,weight_kg,gender,handedness,family_history_any,family_history_first_degree,alcohol_effect_on_tremor,duplicate_patient_id
0,1,PADS,Healthy,Healthy Control,0,56.0,NaN,173.0,78.0,Male,Right,Yes,Yes,Unknown,False
1,2,PADS,Other Movement Disorders,Other Movement Disorder,2,81.0,69.0,193.0,104.0,Male,Right,No,Unknown,No Effect,False
2,3,PADS,Healthy,Healthy Control,0,45.0,NaN,170.0,78.0,Female,Right,No,Unknown,Unknown,False
3,4,PADS,Parkinson's,Parkinson's Disease,1,67.0,63.0,161.0,90.0,Female,Right,No,Unknown,No Effect,False
4,5,PADS,Parkinson's,Parkinson's Disease,1,75.0,65.0,172.0,86.0,Male,Left,No,Unknown,Unknown,False


# 8. Cleaned Demographic Table

The cleaned demographic table contains standardized demographic and diagnostic variables suitable for subsequent statistical analysis and machine learning.


In [10]:
# Display the cleaned demographic table
clean_df.head(10)

,patient_id,study_id,condition_original,condition_group,label,age,age_at_diagnosis,height_cm,weight_kg,gender,handedness,family_history_any,family_history_first_degree,alcohol_effect_on_tremor,duplicate_patient_id
0,1,PADS,Healthy,Healthy Control,0,56.0,NaN,173.0,78.0,Male,Right,Yes,Yes,Unknown,False
1,2,PADS,Other Movement Disorders,Other Movement Disorder,2,81.0,69.0,193.0,104.0,Male,Right,No,Unknown,No Effect,False
2,3,PADS,Healthy,Healthy Control,0,45.0,NaN,170.0,78.0,Female,Right,No,Unknown,Unknown,False
3,4,PADS,Parkinson's,Parkinson's Disease,1,67.0,63.0,161.0,90.0,Female,Right,No,Unknown,No Effect,False
4,5,PADS,Parkinson's,Parkinson's Disease,1,75.0,65.0,172.0,86.0,Male,Left,No,Unknown,Unknown,False
5,6,PADS,Parkinson's,Parkinson's Disease,1,72.0,60.0,171.0,115.0,Female,Right,No,No,Unknown,False
6,7,PADS,Other Movement Disorders,Other Movement Disorder,2,74.0,73.0,181.0,94.0,Male,Right,No,Unknown,No Effect,False
7,8,PADS,Parkinson's,Parkinson's Disease,1,73.0,65.0,168.0,65.0,Female,Right,No,Unknown,No Effect,False
8,9,PADS,Parkinson's,Parkinson's Disease,1,47.0,35.0,184.0,85.0,Male,Left,No,Unknown,Unknown,False
9,10,PADS,Parkinson's,Parkinson's Disease,1,56.0,47.0,187.0,77.0,Male,Right,No,Unknown,Unknown,False


# 9. Exporting the Cleaned Data

The cleaned demographic table and quality assurance summary are exported to the `data/processed` folder.


In [11]:
# Save cleaned demographic table
clean_file = output_folder / "demographics_clean.csv"
clean_df.to_csv(clean_file, index=False)

# Create QA summary
summary = pd.DataFrame({
    "Check": [
        "Total records",
        "Unique participant IDs",
        "Duplicate participant records",
        "Missing age",
        "Missing age at diagnosis",
        "Missing height",
        "Missing weight",
    ],
    "Count": [
        len(clean_df),
        clean_df["patient_id"].nunique(),
        clean_df["duplicate_patient_id"].sum(),
        clean_df["age"].isna().sum(),
        clean_df["age_at_diagnosis"].isna().sum(),
        clean_df["height_cm"].isna().sum(),
        clean_df["weight_kg"].isna().sum(),
    ],
})

summary_file = output_folder / "demographics_qa_summary.csv"
summary.to_csv(summary_file, index=False)

print("Demographic cleaning completed successfully.")
print(f"Clean demographic dataset saved to: {clean_file}")
print(f"QA summary saved to: {summary_file}")

Demographic cleaning completed successfully.
Clean demographic dataset saved to: ..\data\processed\demographics_clean.csv
QA summary saved to: ..\data\processed\demographics_qa_summary.csv


In [12]:
# Display quality assurance summary
summary

,Check,Count
0,Total records,469
1,Unique participant IDs,469
2,Duplicate participant records,0
3,Missing age,0
4,Missing age at diagnosis,98
5,Missing height,1
6,Missing weight,0


# 10. Conclusion

The demographic data were cleaned by standardizing categorical variables, validating numerical values, creating consistent diagnostic groups, and flagging duplicate participant identifiers. Sensitive free-text comments were excluded from the modelling dataset.

The resulting `demographics_clean.csv` file provides a consistent demographic table for later analysis, while `demographics_qa_summary.csv` documents key quality-assurance results.
